In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.diagrams import PersistenceEntropy, PairwiseDistance
from gtda.diagrams import Silhouette
from gtda.diagrams import BettiCurve 
from gtda.homology import VietorisRipsPersistence
from gtda.time_series import TakensEmbedding, SlidingWindow
import plotly.express as px
from sklearn.metrics import mutual_info_score
from gtda.metaestimators import CollectionTransformer
from gtda.pipeline import Pipeline
from gtda.plotting import plot_heatmap

In [ ]:
slug_17df = pd.read_csv('processed_AA-17.csv')
slug_17df

In [ ]:
slug_july = slug_17df[(slug_17df['TimeStamp'] > '2022-07-19 00:00:00') & (slug_17df['TimeStamp'] < '2022-07-24 18:30:00')]


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=4, cols=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['TH_Pressure'],  name="TH_Pressure"), row=1, col=1)
fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['FlowLine_Pressure'],  name="FlowLine_Pressure"), row=1, col=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['GL_Flowrate'],  name="GL_Flowrate"), row=2, col=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['GL_Pressure'],  name="GL_Pressure"), row=3, col=1)

fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['BH_Temperature'],  name="BH_Temperature"), row=4, col=1)
fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['BH_Pressure'],  name="BH_Pressure"), row=4, col=1)
fig.append_trace(go.Scatter(x=slug_july["TimeStamp"], y=slug_july['WCV_Status'],  name="WCV_Status"), row=4, col=1)


fig.update_layout(height=600, width=1200, title_text="Signal Analysis")
fig.show()

In [ ]:
slug_17ts = slug_17df[['TimeStamp', 'TH_Pressure']]
slug_17ts['TimeStamp'] = pd.to_datetime(slug_17ts['TimeStamp'])

slug_17ts

In [ ]:
figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_17ts["TimeStamp"], y=slug_17ts['TH_Pressure'],  name="FHP_wellAA17")

figure_FHP.show()

Of the whole 6 months of data, I am going to focus on few days in July, where slugging was more inconsistent (on/off), so I can catch better if the identification works or not.  

In [ ]:
slug_july = slug_17ts[(slug_17ts['TimeStamp'] > '2022-07-19 00:00:00') & (slug_17ts['TimeStamp'] < '2022-07-27 12:00:00')]

figure_FHP = px.scatter()
figure_FHP.add_scatter(x=slug_july["TimeStamp"], y=slug_july['TH_Pressure'],  name="FHP_wellAA17")
figure_FHP.show()

From previous experience, a time resolution of 1 minute is not enought to identify slugs with topological data analysis (TDA) and persistent homology. \
I am therefore interpolating to a 6s resolution (10 times) on a cubic spline interpolation. 

In [ ]:
from scipy.interpolate import CubicSpline

xmin = 1
xmax = len(slug_july['TH_Pressure'])-1
some_step = 0.1                         # this is a 6 seconds step, 10 points per minute
cs = CubicSpline(range(len(slug_july)), slug_july['TH_Pressure'])
x_range = np.arange(xmin, xmax, some_step)

slug_july.insert(1, "idx", np.linspace(0, len(slug_july)-1, num=len(slug_july)), True)

print(cs(x_range).shape)

# upsampled to 10 times

new_range = pd.date_range(slug_july['TimeStamp'].iloc[0], slug_july['TimeStamp'].iloc[-1], freq='6S')
new_range = new_range[0:len(cs(x_range))]

slug_july_spline = []
slug_july_spline = pd.DataFrame(data=cs(x_range)) 
slug_july_spline['TimeStamp'] = new_range
slug_july_spline


Now splitting the signal into windows of 30 minutes each, each one starting 20 minutes from each starting point. \
Then I am going to analyze each window and see if the time window is slugging or not. 


In [ ]:
from gtda.time_series import SlidingWindow, TakensEmbedding

signal = slug_july_spline[0]
print(len(signal))

window_size = 300 
window_stride = 200
SW = SlidingWindow(size=window_size, stride=window_stride)

X_windows = SW.fit_transform(signal)

In [ ]:
def batch_analyzer(input_df, stride, max_embedding_dimension, max_time_delay):
    max_time_delay = int(max_time_delay)
    max_embedding_dimension = int(max_embedding_dimension)

    i = 0
    averages = np.zeros(shape=(len(input_df),2))
    
   # print('max time delay:',max_time_delay, 'max dim:',max_embedding_dimension)
    
    for timeserie in input_df:
        
        optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
            timeserie, max_time_delay, max_embedding_dimension, stride=stride
            )
        if optimal_embedding_dimension < 3:
            optimal_embedding_dimension = 3
            
        # print('analyzing nr.',i, 'dim',optimal_embedding_dimension,'delay', optimal_time_delay)
        averages[i] = [optimal_embedding_dimension, optimal_time_delay]
        i += 1

    print('Max Dimension:', averages.max(0))
    print('Mean Dimension:', averages.mean(0))
    return averages

In [ ]:
stats = batch_analyzer(X_windows, 1, 13, 20)


By splitting the signal in time windows of 30 minutes (300 points) each starting 20 minutes after the beginning of the preceding one (so to have 10 min overlap) it looks like the optimal embedding parameters are somewhat close to $\tau$=20 and dim=10 (better to be larger than underestimating).

In [ ]:
optimal_time_delay = 20
optimal_embedding_dimension = 10
stride = 1

from gtda.pipeline import Pipeline
from gtda.metaestimators import CollectionTransformer
from gtda.diagrams import PersistenceEntropy

TE = TakensEmbedding(time_delay=optimal_time_delay, dimension=optimal_embedding_dimension, stride=1)
homology_dimensions = (0, 1)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)
PE = PersistenceEntropy()
PE_norm = PersistenceEntropy(normalize=True)
Betti = BettiCurve()

slugging_signals = X_windows
slugging_point_cloud  = TE.fit_transform(slugging_signals)
slugging_diagrams = VRP.fit_transform(slugging_point_cloud)
slugging_entropy = PE.fit_transform(slugging_diagrams)
slugging_entropynorm = PE_norm.fit_transform(slugging_diagrams)
slugging_Betti = Betti.fit_transform(slugging_diagrams)

In [ ]:
import statistics as stats

slug_july_spline['PersH0'] = np.NaN
slug_july_spline['PersH1'] = np.NaN
slug_july_spline['Entropy0'] = np.NaN
slug_july_spline['Entropy1'] = np.NaN
slug_july_spline['signal_max'] = np.NaN
slug_july_spline['signal_min'] = np.NaN
slug_july_spline['signal_avg'] = np.NaN
slug_july_spline['signal_std'] = np.NaN

# sending timestamp to index

slug_july_spline.set_index('TimeStamp', inplace=True) # set column to index

#computing highest persistence 

pers_H0 = []
pers_H1 = []
max_pers_H0 = []
max_pers_H1 = []

for PersDiag in slugging_diagrams: 
    for point in PersDiag:
        birth = point[0]
        death = point[1]
        dimension = point[2]
        persistence = abs(death - birth)
        pers_H0.extend([persistence] if dimension == 0 else [])
        pers_H1.extend([persistence] if dimension == 1 else [])
    max_pers_H0.append(np.amax(pers_H0))
    max_pers_H1.append(np.amax(pers_H1))
    pers_H0 = []
    pers_H1 = []

# Setting thresholds for slugging identification (arbitrary)


    
slug_july_spline

In [ ]:
slug_july_spline.fillna(0.0, inplace=True)

limit_PersH0 = 0.5
limit_PersH1 = 1.5

i = 0 
for time in slug_july_spline.index[::200]:
    if i == len (X_windows):
        break
    if i != 0: 
        #print (slug_july_spline['signal_max'].loc[time] )
        slug_july_spline['signal_max'].loc[time] = max(X_windows[i])
        #print (slug_july_spline['TimeStamp'].loc[time])
        slug_july_spline['signal_min'].loc[time] = min(X_windows[i])
        slug_july_spline['signal_avg'].loc[time] = stats.mean(X_windows[i])
        slug_july_spline['signal_std'].loc[time] = stats.stdev(X_windows[i])
        # setting high value just for visibility and graphing
        if max_pers_H0[i] > limit_PersH0: 
            slug_july_spline['PersH0'].loc[time] = 15.0
            #print(time, slug_july_spline['PersH0'][time])
        if max_pers_H1[i] > limit_PersH1: 
            slug_july_spline['PersH1'].loc[time] = 15.0
            #print(time, slug_july_spline['PersH0'][time])

    i += 1


In [ ]:
figBetti = px.line()
#figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['PersH0'],  name="Pers H0")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['PersH1'],  name="Pers H1")
figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline['signal_std'],  name="std dev")

figBetti.add_scatter(x=slug_july_spline.index, y=slug_july_spline[0],  name="FHP_well10")

When it comes to FFT and Frequency analysis:

In [ ]:
from scipy.signal import periodogram

fs = 1/6            # sampling every 6 seconds
dominant_freq = []

for signal in X_windows:
    f, Pxx_den = periodogram(signal, fs)
    dominant_freq.append(f[np.argmax(Pxx_den)])

figDominant = px.line()
figDominant.add_scatter(y=dominant_freq)

slug_july_spline['Freq(Hz)'] = np.NaN

i = 0 
for time in slug_july_spline.index[::200]:
    if i == len(X_windows):
        break
    if i != 0:
        #print(dominant_freq[i])
        slug_july_spline['Freq(Hz)'][time] = dominant_freq[i]
    i += 1

slug_july_spline.fillna(0.0, inplace=True)

figFFT = px.line()
figFFT.add_scatter(x=slug_july_spline.index, y=slug_july_spline['Freq(Hz)']*1000,  name="Dominant Freq")
#figFFT.add_scatter(x=slug_july_spline.index, y=slug_july_spline['signal_std'],  name="std dev")

figFFT.add_scatter(x=slug_july_spline.index, y=slug_july_spline[0],  name="FHP_well10")

In [ ]:
slug_july_spline[::200]